In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
model = AutoModelForCausalLM.from_pretrained(
        pretrained_model_name_or_path="./Qwen3-0.6B",
        device_map="auto"
    )

model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [3]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="./Qwen3-0.6B")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer

Qwen2Tokenizer(name_or_path='./Qwen3-0.6B', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=False, lstrip=Fa

In [4]:
from IPython.display import display, update_display
from transformers import TextStreamer  # 导入基础流处理器
import torch
from typing import Any

# ---------------------- 【核心：自定义 Notebook 流式输出器】 ----------------------
class NotebookStreamer(TextStreamer):
    def __init__(self, tokenizer, skip_prompt: bool = True, **decode_kwargs: Any):
        super().__init__(tokenizer, skip_prompt, **decode_kwargs)
        self.tokenizer = tokenizer
        self.skip_prompt = skip_prompt  # 保留你原有的跳过prompt功能
        self.full_text = ""
        # 创建一个固定的显示句柄，实时更新
        self.display_handle = display(self.full_text, display_id="model_stream")

    def put(self, output_ids: torch.Tensor):
        """模型每生成一个token，就会调用这个方法"""
        # 跳过提示词的token（和原TextStreamer逻辑一致）
        if self.skip_prompt:
            self.skip_prompt = False
            return

        # 解码token并拼接
        token = self.tokenizer.decode(output_ids, skip_special_tokens=True)
        self.full_text += token

        # 【关键】实时更新Notebook输出（PyCharm完美兼容）
        update_display(self.full_text, display_id=self.display_handle.display_id)

    def end(self):
        """生成结束时调用"""
        pass

In [5]:
messages = [
    {"role" : "user", "content" : "Solve (x + 2)^2 = 0."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 256, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = NotebookStreamer(tokenizer, skip_prompt = True),
)

"<think>\nOkay, so I need to solve the equation (x + 2)^2 = 0. Hmm, let me think. I remember that when you have something squared equals zero, the solution is the number that, when you square it, gives you zero. But wait, how exactly do I approach this?\n\nFirst, let me write down the equation again to make sure I have it right: (x + 2)^2 = 0. Yeah, that's correct. So, the square of something equals zero. That means that the something itself must be zero because if you square a number and get zero, the number has to be zero. Is that right? Let me verify.\n\nSuppose I have (x + 2)^2 = 0. If I take the square root of both sides, I get x + 2 = ±√0. And ��0 is zero, so then x + 2 = 0 or x + 2 = 0. Wait, is that correct? Let me think again. The square root of 0 is 0, so taking square roots of both sides gives x + 2 = ±0. So, solving for x, it would be x + 2 = 0, which gives x = -2. But hold on, if I take square roots, sometimes there can be multiple solutions, but in this case, since the sq

In [6]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = True, # Enable thinking
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 2048, # Increase for longer outputs!
    temperature = 0.6, top_p = 0.95, top_k = 20, # For thinking
    streamer = NotebookStreamer(tokenizer, skip_prompt = True),
)

"<think>\nOkay, so I need to solve the equation (x + 2)^2 = 0. Hmm, let me think. I remember that when you have something squared equals zero, the solution is the number that, when you square it, gives you zero. But wait, how exactly do I approach this?\n\nFirst, let me write down the equation again to make sure I have it right: (x + 2)^2 = 0. Yeah, that's correct. So, the square of something equals zero. That means that the something itself must be zero because if you square a number and get zero, the number has to be zero. Is that right? Let me verify.\n\nSuppose I have (x + 2)^2 = 0. If I take the square root of both sides, I get x + 2 = ±√0. And ��0 is zero, so then x + 2 = 0 or x + 2 = 0. Wait, is that correct? Let me think again. The square root of 0 is 0, so taking square roots of both sides gives x + 2 = ±0. So, solving for x, it would be x + 2 = 0, which gives x = -2. But hold on, if I take square roots, sometimes there can be multiple solutions, but in this case, since the sq

In [7]:
from datasets import load_dataset

In [8]:
reasoning_dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot", cache_dir = "./cache")
reasoning_dataset

Dataset({
    features: ['expected_answer', 'problem_type', 'problem_source', 'generation_model', 'pass_rate_72b_tir', 'problem', 'generated_solution', 'inference_mode'],
    num_rows: 19252
})

In [9]:
non_reasoning_dataset = load_dataset("mlabonne/FineTome-100k", split = "train", cache_dir = "./cache")
non_reasoning_dataset

Dataset({
    features: ['conversations', 'source', 'score'],
    num_rows: 100000
})

In [10]:
def generate_reasoning_conversation(examples):
    problems  = examples["problem"]
    solutions = examples["generated_solution"]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append([
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : solution},
        ])
    return { "conversations": conversations, }

In [11]:
reasoning_conversations = tokenizer.apply_chat_template(
    list(reasoning_dataset.map(generate_reasoning_conversation, batched = True)["conversations"]),
    tokenize = False,
)

Map:   0%|          | 0/19252 [00:00<?, ? examples/s]

In [12]:
reasoning_conversations[0]

"<|im_start|>user\nGiven $\\sqrt{x^2+165}-\\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.\n\nFirst, let me write down the equation again to make sure I have it right:\n\n√(x² + 165) - √(x² - 52) = 7.\n\nOkay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:\n\n√(x² + 165) = 7 + √(x² - 52).\n\nNow, if I square both sides, maybe I can get rid of the square roots. Let's do that:\n\n(√(x² + 165))² = (7 + √(x² - 52))².\n\nSimplifying the left side:\n\nx² + 165 = 49 + 14√(x² - 52) + (√(x² - 52))².\n\nThe right side is expanded using the formula (a + b)² = a² + 2ab + b². So the right side becomes 7² + 2*7*√(x² - 52) + (√(x² 

In [13]:
def generate_non_reasoning_conversation(examples):
    problems  = [conversation[0]["value"] for conversation in examples["conversations"]]
    solutions = [conversation[1]["value"] for conversation in examples["conversations"]]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append([
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : solution},
        ])
    return { "conversations": conversations, }

In [14]:
non_reasoning_conversations = tokenizer.apply_chat_template(
    list(non_reasoning_dataset.map(generate_non_reasoning_conversation, batched = True)["conversations"]),
    tokenize = False,
)

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [15]:
non_reasoning_conversations[0]

'<|im_start|>user\nExplain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</th

In [16]:
print(f"Chat Percentage: {len(non_reasoning_conversations)/(len(reasoning_conversations)+len(non_reasoning_conversations))}")

Chat Percentage: 0.8385603595746822


In [17]:
chat_percentage = 0.75

In [18]:
import pandas as pd

num_non_reasoning_samples = len(reasoning_conversations)/(1-chat_percentage)-len(reasoning_conversations)

non_reasoning_subset = pd.Series(non_reasoning_conversations)
non_reasoning_subset = non_reasoning_subset.sample(
    int(num_non_reasoning_samples),
    random_state = 42,
)

data = pd.concat([
    pd.Series(reasoning_conversations),
    pd.Series(non_reasoning_subset)
])
data.name = "text"

In [19]:
from datasets import Dataset

combined_dataset = Dataset.from_pandas(pd.DataFrame(data))
combined_dataset = combined_dataset.shuffle(seed = 42)
combined_dataset

Dataset({
    features: ['text', '__index_level_0__'],
    num_rows: 77008
})

In [20]:
import os
import json

os.makedirs("./dataset", exist_ok = True)
with open("./dataset/combined_dataset.json", "w") as f:
    json.dump(combined_dataset.to_dict(), f)